# Postgres + PaperRepository — service test

Tests DB connection, upsert idempotency, and reads.

**Prereqs:** `docker compose up -d postgres`

In [ ]:
# ⚠️ GOTCHA: Paper MUST be imported before make_database(),
# otherwise create_all doesn't see the table (ORM registration order).
from src.models.paper import Paper
from src.db.factory import make_database

db = make_database()
print('connected')

In [ ]:
# Count + peek at stored papers
from src.repositories.paper import PaperRepository

with db.get_session() as session:
    repo = PaperRepository(session)
    papers = session.query(Paper).limit(5).all()
    total = session.query(Paper).count()

print(f'total papers in DB: {total}')
for p in papers:
    print(p.arxiv_id, '|', (p.title or '')[:60], '| raw_text:', len(p.raw_text or ''), 'chars')

In [ ]:
# Upsert idempotency check: same arxiv_id twice → one row, no duplicate
from src.schemas.arxiv.paper import PaperCreate

test = PaperCreate(arxiv_id='test.00001', title='Idempotency test', abstract='x')

with db.get_session() as session:
    repo = PaperRepository(session)
    a = repo.upsert(test)
    b = repo.upsert(test)   # second call → UPDATE, not INSERT
    count = session.query(Paper).filter_by(arxiv_id='test.00001').count()

print('rows for test.00001:', count, '(expect 1)')

In [ ]:
# Cleanup the test row
with db.get_session() as session:
    session.query(Paper).filter_by(arxiv_id='test.00001').delete()
    session.commit()
print('cleaned up')